In [11]:
import pandas as pd
import sqlite3
from pathlib import Path

In [14]:
folder = Path("../data/raw")
db_path = "../data/processed/olist.db"
with sqlite3.connect(db_path) as conn:
    for file in folder.glob("*.csv"):
        print(f"Processing: {file.name}...")

        df = pd.read_sql
        df = pd.read_csv(file)

        table_name = file.stem

        clean_table_name = table_name.removeprefix("olist_").removesuffix("_dataset")

        df.to_sql(
            name=clean_table_name,
            con=conn,
            if_exists="append",
            index=False
        )
print("All files have been read.")

Processing: olist_sellers_dataset.csv...
Processing: product_category_name_translation.csv...
Processing: olist_orders_dataset.csv...
Processing: olist_order_items_dataset.csv...
Processing: olist_customers_dataset.csv...
Processing: olist_geolocation_dataset.csv...
Processing: olist_order_payments_dataset.csv...
Processing: olist_order_reviews_dataset.csv...
Processing: olist_products_dataset.csv...
All files have been read.


In [18]:
# Check tables exist in the table
conn = sqlite3.connect("../data/processed/olist.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in database:", tables)

conn.close()

Tables in database: [('sellers',), ('product_category_name_translation',), ('orders',), ('order_items',), ('customers',), ('geolocation',), ('order_payments',), ('order_reviews',), ('products',)]


In [21]:
# Check all table columns to find key
for file in folder.glob("*.csv"):
    print(f"{file.name} columns:")
    df = pd.read_csv(file)
    print(list(df))
    print("\n")

olist_sellers_dataset.csv columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


product_category_name_translation.csv columns:
['product_category_name', 'product_category_name_english']


olist_orders_dataset.csv columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


olist_order_items_dataset.csv columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


olist_customers_dataset.csv columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


olist_geolocation_dataset.csv columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


olist_order_payments_dataset.csv columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', '

In [23]:
conn = sqlite3.connect("../data/processed/olist.db")

def q(sql):
    return pd.read_sql(sql, conn)

In [27]:
# Order status breakdown
q("""SELECT order_status,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY orders DESC;
""")

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [34]:
# Date range
q("""SELECT MIN(order_purchase_timestamp) AS first_order, MAX(order_purchase_timestamp) AS last_order
FROM orders;
""")

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [36]:
# orders per month 
q("""SELECT strftime('%Y-%m', order_purchase_timestamp) AS month, COUNT(*) AS orders
FROM orders
GROUP BY month
ORDER BY month;
""")

,month,orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [38]:
# Check NULLs
q("""
SELECT
  COUNT(*) AS total_orders,
  SUM(order_approved_at IS NULL) AS null_approved,
  SUM(order_delivered_carrier_date IS NULL) AS null_carrier,
  SUM(order_delivered_customer_date IS NULL) AS null_delivered,
  SUM(order_estimated_delivery_date IS NULL) AS null_estimated
FROM orders;
""")

,total_orders,null_approved,null_carrier,null_delivered,null_estimated
0,99441,160,1783,2965,0


In [42]:
q("""SELECT order_status,
       COUNT(*) AS orders,
       SUM(order_delivered_customer_date IS NULL) AS null_delivery_date
FROM orders
GROUP BY order_status;
""")

,order_status,orders,null_delivery_date
0,approved,2,2
1,canceled,625,619
2,created,5,5
3,delivered,96478,8
4,invoiced,314,314
5,processing,301,301
6,shipped,1107,1107
7,unavailable,609,609


In [44]:
# Check for duplicate orders
q("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT order_id) AS distinct_orders
FROM orders;
""")

,total_rows,distinct_orders
0,99441,99441


In [45]:
# Orders with multiple items
q("""
SELECT COUNT(*) AS total_item_rows,
       COUNT(DISTINCT order_id) AS orders_with_items
FROM order_items;
""")

,total_item_rows,orders_with_items
0,112650,98666


In [46]:
# Orders with multiple payment rows
q("""
SELECT COUNT(*) AS orders_with_multiple_payments
FROM (
  SELECT order_id
  FROM order_payments
  GROUP BY order_id
  HAVING COUNT(*) > 1
);
""")

,orders_with_multiple_payments
0,2961


In [48]:
q("""
SELECT COUNT(*) AS orders_with_multiple_reviews
FROM (
  SELECT order_id
  FROM order_reviews
  GROUP BY order_id
  HAVING COUNT(*) > 1
);
""")

,orders_with_multiple_reviews
0,547


In [50]:
q("""
SELECT COUNT(*) FROM orders;
""")

,COUNT(*)
0,99441


In [51]:
# Joining created extra rows
q("""
SELECT COUNT(*)
FROM orders o
JOIN order_payments p ON o.order_id = p.order_id;
""")

,COUNT(*)
0,103886
